# Практикум 2. Подготовка наборов данных

Сопровождает интерактивную лабораторию №2 («Подготовка наборов данных»). Тема курса: формирование выборок, разметка и качество данных

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.5, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Разбиение train / val / test со стратификацией

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, accuracy_score

train_val, test = train_test_split(df, test_size=0.2, stratify=df.label, random_state=0)
train, val = train_test_split(train_val, test_size=0.25, stratify=train_val.label, random_state=0)
print(len(train), len(val), len(test))
pd.DataFrame({"train": train.label.value_counts(), "val": val.label.value_counts(), "test": test.label.value_counts()})

Если данные отсортированы по классам, а перемешивания нет, тест состоит из одного класса:

In [ ]:
srt = df.sort_values("label").reset_index(drop=True)
cut = int(len(srt) * 0.8)
print("Тест без перемешивания:", srt.label[cut:].value_counts().to_dict())

## 2. Утечка данных
Копии обучающих текстов в тесте завышают оценку.

In [ ]:
vec = TfidfVectorizer().fit(train.text)
def acc_1nn(tr, te):
    m = KNeighborsClassifier(1, metric="cosine").fit(vec.transform(tr.text), tr.label)
    return accuracy_score(te.label, m.predict(vec.transform(te.text)))
leak = train.sample(len(test) // 2, random_state=0)
test_leak = pd.concat([test.iloc[: len(test) // 2], leak])
print("Честный тест:", round(acc_1nn(train, test), 3), "| тест с утечкой:", round(acc_1nn(train, test_leak), 3))
print("Общих текстов train/test:", len(set(train.text) & set(test.text)), "| с утечкой:", len(set(train.text) & set(test_leak.text)))

## 3. Качество разметки: согласие разметчиков (κ Коэна)

In [ ]:
rng = np.random.default_rng(0)
classes = np.array(sorted(df.label.unique()))
def annotate(labels, p):
    labels = np.array(labels).copy()
    flip = rng.random(len(labels)) < p
    labels[flip] = [rng.choice(classes[classes != l]) for l in labels[flip]]
    return labels
Xtr, Xte = vec.transform(train.text), vec.transform(test.text)
rows = []
for p in [0, 0.1, 0.2, 0.3, 0.4]:
    a, b = annotate(train.label, p), annotate(train.label, p)
    clf = LogisticRegression(max_iter=1000).fit(Xtr, a)
    rows.append((p, cohen_kappa_score(a, b), accuracy_score(test.label, clf.predict(Xte))))
res = pd.DataFrame(rows, columns=["доля ошибок разметчика", "kappa двух разметчиков", "accuracy на чистом тесте"])
display(res.round(3))
res.plot(x="доля ошибок разметчика", y=["kappa двух разметчиков", "accuracy на чистом тесте"], marker="o")
plt.show()

## Задания

1. Найдите в корпусе точные дубликаты (`df.duplicated`) и удалите их до разбиения. Почему это нужно делать до, а не после?
2. Постройте разбиение без `stratify` на очень маленьком редком классе (оставьте 10 документов). Что произойдёт с редким классом в тесте?
3. Придумайте правила инструкции для разметчиков, которые повысили бы κ на неоднозначных текстах.